In [ ]:
# ============================================================
# Cell 8: 분류 모델 정의 (Weighted Pooling + Residual Head + Partial Unfreeze + Post-LN)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

# 잔차 연결 블록 (깊은 층에서 핵심 키워드 정보 소실 방지)
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout_rate=0.3):
        super().__init__()
        self.fc = nn.Linear(dim, dim)
        self.ln = nn.LayerNorm(dim)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, x):
        residual = x # 원본 정보 저장
        out = self.fc(x)
        out = self.ln(out)
        out = self.gelu(out)
        out = self.dropout(out)
        return out + residual # 우회 도로(잔차) 더하기

class ConversationClassifier(nn.Module): 
    def __init__(self, model_name, num_classes=5, dropout_rate=0.3, freeze_backbone=False): 
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size # 768
        
        # [핵심 1] Partial Unfreeze: 전체를 얼리되, 마지막 2개 층만 열어 미세 뉘앙스 학습
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
            for param in self.backbone.encoder.layer[-2:].parameters():
                param.requires_grad = True
            print("❄️ 백본 가중치 고정됨 (단, 마지막 2개 층은 Partial Unfreeze 적용)")
            
        # [핵심 2] Weighted Layer Pooling 가중치 (임베딩 1 + 히든 12 = 총 13층)
        self.layer_weights = nn.Parameter(torch.ones(13))
        
        # 헤드 입력 차원 (Mean 768 + Max 768 = 1536)
        input_dim = hidden_size * 2  
        
        # [핵심 3] 깊은 헤드 구조 (풀링 직후 정규화를 없애고, Linear 통과 후 정규화)
        self.fc1 = nn.Linear(input_dim, 512)
        self.layer_norm1 = nn.LayerNorm(512) # 🌟 1536 생날것을 섞은 후 여기서 정규화!
        self.activation1 = nn.GELU()
        self.dropout1 = nn.Dropout(dropout_rate)
        
        # [핵심 4] 잔차 연결 블록 삽입
        self.res_block = ResidualBlock(512, dropout_rate)
        
        # 두 번째 은닉층
        self.fc2 = nn.Linear(512, 256)
        self.layer_norm2 = nn.LayerNorm(256)
        self.activation2 = nn.GELU()
        self.dropout2 = nn.Dropout(dropout_rate)
        
        # 최종 로짓 출력층
        self.fc3 = nn.Linear(256, num_classes)
    
    def pool(self, all_hidden_states, attention_mask):
        # 13개 층을 학습된 가중치 비율로 섞음
        weights = F.softmax(self.layer_weights, dim=0)
        weighted_hidden = sum(w * h for w, h in zip(weights, all_hidden_states))
        
        mask_expanded = attention_mask.unsqueeze(-1).float()
        
        # Mean Pooling (문맥)
        sum_embeddings = torch.sum(weighted_hidden * mask_expanded, dim=1)
        sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
        mean_pooled = sum_embeddings / sum_mask
        
        # Max Pooling (핵심 키워드 에너지)
        masked_hidden = weighted_hidden.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e9)
        max_pooled = torch.max(masked_hidden, dim=1)[0]
        
        # 🌟 정규화 없이 순수한 에너지를 그대로 이어붙임
        return torch.cat([mean_pooled, max_pooled], dim=1)
    
    def forward(self, input_ids, attention_mask):
        # output_hidden_states=True로 전 층의 아웃풋을 받음
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        
        # 커스텀 Pooler 통과 (1536차원 순수 벡터 생성)
        pooled_output = self.pool(outputs.hidden_states, attention_mask)
        
        # 1층: Linear 믹싱 후 정규화
        hidden = self.fc1(pooled_output)
        hidden = self.layer_norm1(hidden) # 🌟 사용자의 통찰이 담긴 진짜 꿀자리
        hidden = self.activation1(hidden)
        hidden = self.dropout1(hidden)
        
        # 2층: 잔차 블록 통과
        hidden = self.res_block(hidden)
        
        # 3층: 압축 및 정규화
        hidden = self.fc2(hidden)
        hidden = self.layer_norm2(hidden)
        hidden = self.activation2(hidden)
        hidden = self.dropout2(hidden)
        
        # 출력
        logits = self.fc3(hidden)
        return logits

# --- 모델 인스턴스 생성 ---
model = ConversationClassifier(
    model_name=MODEL_NAME,
    num_classes=len(label2id),
    dropout_rate=DROPOUT_RATE,
    freeze_backbone=True
).to(device)

# --- 파라미터 수 확인 ---
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n{"="*50}')
print(f'🤖 모델 인스턴스 생성 완료 (최종 병기 아키텍처 V2)')
print(f'전체 파라미터: {total_params:,}개')
print(f'학습 가능 파라미터: {trainable_params:,}개 ({trainable_params/total_params*100:.2f}%)')
print(f'{"="*50}\n')


In [ ]:
# ============================================================
# Cell 9: 옵티마이저, 스케줄러, 손실 함수 설정 (V2 대응)
# ============================================================
import torch
import torch.nn as nn
from transformers import get_linear_schedule_with_warmup

# --- 1. 파라미터 분리 및 학습 대상 선별 ---
head_params = []
backbone_params = []

for name, param in model.named_parameters():
    # ❄️ 가중치가 고정된(requires_grad=False) 파라미터는 옵티마이저에서 제외
    if not param.requires_grad:
        continue 
    
    # [중요] 이름에 'backbone'이 들어가지 않은 모든 부품은 '헤드 및 커스텀 레이어'로 분류
    # 여기에는 fc1~3, res_block, layer_weights, layer_norm1~2 등이 모두 포함됩니다.
    if 'backbone' not in name:
        head_params.append(param)
    else:
        #requires_grad가 True이면서 backbone인 것 (즉, Partial Unfreeze된 10, 11번 레이어)
        backbone_params.append(param)

# 🌟 차등 학습률(Discriminative LR) 설정
# 백본은 이미 완성된 지식이므로 아주 천천히(1/10) 학습하고, 헤드는 새롭게 배우므로 원래 속도로 학습
BACKBONE_LEARNING_RATE = LEARNING_RATE / 10  
HEAD_LEARNING_RATE = LEARNING_RATE           

optimizer_grouped_parameters = [
    {'params': backbone_params, 'lr': BACKBONE_LEARNING_RATE, 'weight_decay': WEIGHT_DECAY},
    {'params': head_params, 'lr': HEAD_LEARNING_RATE, 'weight_decay': WEIGHT_DECAY}
]

# --- 2. 옵티마이저: AdamW ---
optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

# --- 3. 스케줄러: Linear Warmup + Decay ---
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# --- 4. 손실 함수: Label Smoothing 적용 (과적합 방지 및 일반화 성능 향상) ---
# 0.1 정도의 스무딩을 주면 모델이 너무 오만하게 확신하지 않아 6에폭 뻥크를 막는 데 효과적입니다.
criterion = nn.CrossEntropyLoss(label_smoothing=0.15)

# --- 5. 설정 검증 출력 ---
print(f'\n{"="*60}')
print(f'🤖 모델 구조 및 학습 파라미터 정보 (최종 병기 V2)')
print(f'{"="*60}')
print(f'Total Parameters   : {total_params:,} 개')
print(f'Trainable Params   : {trainable_params:,} 개 ({trainable_params/total_params*100:.2f}%)')
print(f'-'*60)
print(f'⚙️ 최적화 및 스케줄러 설정')
print(f'-'*60)
print(f'Total steps  : {total_steps:,} 스텝')
print(f'Warmup steps : {warmup_steps:,} 스텝')
print(f'Optimizer    : AdamW (Discriminative LR - 전략적 차등 학습)')
print(f'  - Backbone LR : {BACKBONE_LEARNING_RATE} (마지막 2개 층 미세 조정)')
print(f'  - Head LR     : {HEAD_LEARNING_RATE} (커스텀 헤드 집중 학습)')
print(f'Scheduler    : Linear Warmup + Decay')
print(f'Loss         : CrossEntropyLoss (Label Smoothing 0.1 적용 🌟)')
print(f'{"="*60}\n')
